The current work is to cluster the input activations of the whole cube.  
I would then like to see if this kernel is actually doing anything that can be explained using horizontal slices taken together.   

so right now, Im interested in only 1 POI, (2,4).  

Question: is the slice the most useful component of this activation?
- We cluster the pointwise mults of the cube for this POI, for the same examples we took for now
- Inside the cluster, we would assume that the activations come from the same slice across clusters

- First, I need to get my shit together, what are the next steps?


- Cluster the pointwise mults of ch12 fully for points 2,4 for the same examples as the ones we took for in6 of this cube
- Then we try to see if clusters are dependent only on the slice.  
- Within this, there is something i can look at
  - components that move togehter, these give us activations which are correlated (pearson correlation matrix)
    - If two pixels are correlated in the inputs, then they are always coming together, which is important
  - I can use standard deviation of each componet in the cluster to see if they are similar. We look at the ones which are low
    - Lets do this first


Quite a lot of them are very empty. Now i see some correlation in the 7th channel, will need to look at it.  

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
from pt_to_api.contribs.v1 import show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP

# (2,3) POI analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def _scatter_plot_1d(numbers, suff=""):

    # 2. Create the visualization
    plt.figure(figsize=(10, 2))
    sns.stripplot(x=numbers, color='blue', alpha=0.5, jitter=True)

    plt.title('1D Clustering Visualization' + suff)
    plt.xlabel('Value')
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.show()

In [ ]:
input_aliases = ['4-cYS',
 '4-GT3',
 '4-U12',
 '4-a4C',
 '4-DjO',
 '4-duA',
 '4-T97',
 '4-aof',
 '4-BaZ',
 '4-2Z9',
 '4-lkg',
 '4-1a2',
 '4-JJV',
 '4-Wa5',
 '4-q4M',
 '4-mz3',
 '4-Ds5',
 '4-4AN',
 '4-mev',
 '6-TSz']

sms = list(SaliencyMap.objects.filter(input__alias__in=input_aliases, coordinate="layers.2.out_12"))
acts = {}
for a in input_aliases:
    acts[a] = list(Activation.objects.filter(coordinate__startswith="layers.1.", input__alias=a).order_by("coordinate"))

In [ ]:
weights = list(Weight.objects.filter(coordinate__startswith="layers.2.out_12", data_type="weights").order_by("coordinate"))
_weights = [np.array(w.data) for w in weights]
kernel = np.stack(_weights)

S([kernel[0], _weights[0]])

In [ ]:
numbers = [sm.data[2][3] for sm in sms]
coord = (2, 3)
_scatter_plot_1d(numbers, str(coord))

In [ ]:
k = '4-cYS'

vecs = [
    np.stack([np.array(a.data) for a in acts[k]])
    for k in acts
]
# receptive field of (2,3) = y-start=(2*2 - 1)=3, x-start=5, ends=(6, 8)

patches = [v[:, 3:6, 5:8] for v in vecs]
lin_patches = [p.reshape(-1) for p in patches]
print(patches[0].shape, kernel.shape, lin_patches[0].shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_patches = scaler.fit_transform(lin_patches)
scaled_pws = scaled_patches * kernel.reshape(-1)

In [ ]:
from sklearn.decomposition import PCA

# 7 is fine, found after doing for 20
pca = PCA(7)
X_reduced = pca.fit_transform(scaled_pws)


exp_var_pca = pca.explained_variance_ratio_
cum_sum_eigenvalues = np.cumsum(exp_var_pca)

# 5. Create the Scree Plot
plt.figure(figsize=(10, 6))

# Individual variance bars
plt.bar(range(1, len(exp_var_pca) + 1), exp_var_pca, alpha=0.5, align='center',
        label='Individual explained variance')

# Cumulative variance step plot
plt.step(range(1, len(cum_sum_eigenvalues) + 1), cum_sum_eigenvalues, where='mid',
         label='Cumulative explained variance', color='red')

plt.ylabel('Explained variance ratio')
plt.xlabel('Principal component index')
plt.title('Scree Plot: Explained Variance by Components')
plt.xticks(range(1, len(exp_var_pca) + 1))
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt

# linkage performs the actual clustering
linkage_matrix = sch.linkage(X_reduced, method='single')

plt.figure(figsize=(10, 7))
sch.dendrogram(linkage_matrix)
plt.title('Dendrogram')
plt.xlabel('Samples')
plt.ylabel('Euclidean distances')
plt.show()

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt

# linkage performs the actual clustering
linkage_matrix = sch.linkage(X_reduced, method='ward')

plt.figure(figsize=(10, 7))
sch.dendrogram(linkage_matrix)
plt.title('Dendrogram')
plt.xlabel('Samples')
plt.ylabel('Euclidean distances')
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import pandas as pd

results = []
K_range = range(2, 11) # Checking 2 to 10 clusters
X = X_reduced

for k in K_range:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(X)
    
    # Calculate scores
    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    
    results.append({'k': k, 'silhouette': sil, 'calinski': ch})

# Convert to DataFrame to find the winner
df_results = pd.DataFrame(results)
best_k = df_results.loc[df_results['silhouette'].idxmax(), 'k']
print(f"The optimal number of clusters based on Silhouette is: {best_k}")

df_results

In [ ]:
model = AgglomerativeClustering(n_clusters=5, linkage='ward')
labels = model.fit_predict(X)
np.unique(labels, return_counts=True)

In [ ]:
from collections import defaultdict
l2patch = defaultdict(list)
for i in range(len(labels)):
    l2patch[labels[i]].append(patches[i])

In [ ]:
for k, v in l2patch.items():
    print(k, len(v))

In [ ]:
S([p[2] for p in l2patch[0]], (10,5), ncols=4)

In [ ]:
S([p[2] for p in l2patch[1]], (10,5), ncols=4)

In [ ]:
S([p[6] for p in l2patch[4]], ncols=1)

# For specific inputs

For http://localhost:5173/models/simple_mnist_v1/4-duA/v1/kernel/layers.2/12, i see the slicewise spread being more for channel 2.   
Many of the kernels are quite empty here for the input slice for the data.  